In [0]:
%sql
-- ============================================================
--  Exploración Inicial y Análisisde Calidad
-- ============================================================

-- ============================================================
-- ¿Cuántos registros tenemos?
-- ============================================================

select count(*) as total_registros
from real_estate.bronze.properties_bronze;



In [0]:
%sql
-- ============================================================
-- Explorar estructura de la tabla
-- ============================================================


describe table real_estate.bronze.properties_bronze;

In [0]:
%sql
-- ============================================================
-- Ver muestra de datos (columnas principales)
-- ============================================================

select 
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    estado,
    zona
from real_estate.bronze.properties_bronze
limit 10;

In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

with total as (
    select count(*) as n
    from real_estate.bronze.properties_bronze
),

nulos as (
    select
        count(*) - count(id) as nulos_id,
        count(*) - count(ubicacion) as nulos_ubicacion,
        count(*) - count(precio) as nulos_precios,
        count(*) - count(expensas) as nulos_expensas,
        count(*) - count(tipo_de_operacion) as nulos_tipo_operacion,
        count(*) - count(moneda) as nulos_moneda,
        count(*) - count(ambientes) as nulos_ambientes,
        count(*) - count(metros_cuadrados_totales) as nulos_m2_totales,
        count(*) - count(metros_cuadrados_cubiertos) as nulos_metros_m2_cubiertos,
        count(*) - count(antiguedad) as nulos_antiguedad,
        count(*) - count(estado) as nulos_estado,
        count(*) - count(zona) as nulos_zona
    from real_estate.bronze.properties_bronze

)
select 
    t.n as total_registros,
    n.*
from total t, nulos n;


In [0]:
%sql
-- ============================================================
-- Porcentaje de nulos en columnas clave
-- ============================================================

with totales as (
    select count(*) as total
    from real_estate.bronze.properties_bronze
)
select 
    round((t.total - count(precio)) * 100.0 / t.total, 2) as pct_precios_nulos,
    round((t.total - count(expensas)) * 100.0 / t.total, 2) as pct_expensas_nulos,
    round((t.total - count(ambientes)) * 100.0 / t.total, 2) as pct_ambientes_nulos,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
from real_estate.bronze.properties_bronze
cross join totales t 
group by t.total;

In [0]:
%sql
-- ============================================================
-- Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas


with totales as (
    select count(*) as total
    from real_estate.bronze.properties_bronze
),
pct_nulos as (
select 
    round((t.total - count(precio)) * 100.0 / t.total, 2) as pct_precios_nulos,
    round((t.total - count(expensas)) * 100.0 / t.total, 2) as pct_expensas_nulos,
    round((t.total - count(ambientes)) * 100.0 / t.total, 2) as pct_ambientes_nulos,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
from real_estate.bronze.properties_bronze
cross join totales t 
group by t.total
)
select columna, porcentaje_nulos
from pct_nulos 
unpivot  (
    porcentaje_nulos for columna in (pct_precios_nulos, pct_expensas_nulos, 
        pct_ambientes_nulos, 
        pct_nulos_m2_totales, pct_nulos_m2_cubiertos,
        pct_nulos_orientacion, pct_nulos_antiguedad
    )
)
where porcentaje_nulos > 50
order by porcentaje_nulos desc;

In [0]:
%sql
-- ============================================================
-- Distribución de tipo_de_operacion
-- ============================================================

select 
    tipo_de_operacion,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over (), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by tipo_de_operacion
order by cantidad desc;

In [0]:
%sql
-- ============================================================
-- Distribución de moneda
-- ============================================================

select
    moneda,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by moneda
order by cantidad desc;
    

In [0]:
%sql
-- ============================================================
-- Distribución de ambientes
-- ============================================================

select 
    ambientes,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by ambientes
order by ambientes;

In [0]:
%sql
-- ============================================================
-- Top 15 zonas con más propiedades
-- ============================================================

select 
    zona,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by zona
order by cantidad desc
limit 15;

In [0]:
%sql
-- ============================================================
-- Distribución de estado de las propiedades
-- ============================================================

select 
    estado,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by estado
order by cantidad desc;

In [0]:
%sql
-- ============================================================
-- (ANTES de limpiar): Estadísticas de PRECIO (datos crudos)
-- ============================================================

select 
    moneda,
    count(*) as cantidad,
    round(min(precio), 2) as precio_minimo,
    round(max(precio), 2) as precio_maximo,
    round(avg(precio), 2) as precio_promedio,
    round(percentile(precio, 0.05), 2) as precio_mediana,
    round(percentile(precio, 0.25), 2) as precio_p25,
    round(percentile(precio, 0.75), 2) as precio_p75
from real_estate.bronze.properties_bronze
where precio is not null and precio > 0 
group by moneda
order by cantidad desc;

In [0]:
%sql
-- ============================================================
-- (ANTES de limpiar): Estadísticas de METROS CUADRADOS (datos crudos)
-- ============================================================


select 
    'metros_cuadrados_totales' as columna,
    count(*) as cantidad_no_nulos,
    round(min(metros_cuadrados_totales), 2) as minimo,
    round(max(metros_cuadrados_totales), 2) as maximo,
    round(avg(metros_cuadrados_totales), 2) as promedio,
    round(percentile(metros_cuadrados_totales, 0.05), 2) as mediana
from real_estate.bronze.properties_bronze
where metros_cuadrados_totales is not null and metros_cuadrados_totales > 0

union all 

select 
    'metros_cuadrados_cubiertos' as columna,
    count(*) as cantidad_no_nulos,
    round(min(metros_cuadrados_cubiertos), 2) as minimo,
    round(max(metros_cuadrados_cubiertos), 2) as maximo,
    round(avg(metros_cuadrados_cubiertos), 2) as promedio,
    round(percentile(metros_cuadrados_cubiertos, 0.05), 2) as mediana
from real_estate.bronze.properties_bronze
where metros_cuadrados_cubiertos is not null and metros_cuadrados_cubiertos > 0;

## Es acá donde el Data engineer entra en juego, sin la automatización y el tratamiento adecuado de los datos, el análisis no es viable, o simplemente carece de sentido el resultado

# Corrigamos los datos por ahora de manera temporal y veamos que ocurre

In [0]:
%sql
describe real_estate.bronze.properties_bronze

In [0]:
%sql
create or replace temporary view propiedades_clean as 
select
case 
when precio rlike '^[^a-zA-Z]+$' then precio::double
 else null
end as precio,
 moneda,
case 
 when ambientes rlike '^[^a-zA-Z]+$' then ambientes::double
 else null 
end as ambientes,
case 
  when metros_cuadrados_totales rlike '^[^a-zA-Z]+$' then metros_cuadrados_totales::double
  else null 
end as metros_cuadrados_totales,
case 
  when metros_cuadrados_cubiertos rlike '^[^a-zA-Z]+$' then metros_cuadrados_cubiertos::double
  else null
end as metros_cuadrados_cubiertos,
case 
  when antiguedad rlike '^[^a-zA-Z]+$' then antiguedad::double 
  else null 
end as antiguedad,
id,
ubicacion,
numero,
calle,
expensas,
tipo_de_operacion,
orientacion_cardinal,
orientacion_inmueble,
piso,
cochera,
estado,
tipo_vendedor,
url,
zona,
fecha,
hora
from real_estate.bronze.properties_bronze

## Repetimos el mismo análisis de antes para ver que pasa

In [0]:
%sql
select 
    moneda,
    tipo_de_operacion,
    count(*) as cantidad,
    round(min(precio), 2) as precio_minimo,
    round(max(precio), 2) as precio_maximo,
    round(avg(precio), 2) as precio_promedio,
    round(percentile(precio, 0.05), 2) as precio_mediana,
    round(percentile(precio, 0.25), 2) as precio_p25,
    round(percentile(precio, 0.75), 2) as precio_p75
from propiedades_clean
where precio is not null and precio > 0 
group by moneda, tipo_de_operacion
order by cantidad desc;

## Hallazgo: monedas inválidas, tipos de operación sucios y outliers extremos

Del análisis anterior detectamos:
- **Monedas basura:** MXN, guaranies, uyu, consultar, ar — son errores del scraping (< 15 registros en total)
- **Tipos de operación sucios:** variantes como "alquieler", "alquier", "alguilar", "venta/alquiler", etc.
- **Nulls:** registros sin moneda y sin tipo de operación
- **Outliers extremos:** precios máximos de 1,111,111,111 USD y 1,434,768,228 ARS son claramente errores

**Acción:** Filtramos solo USD y ARS, solo los 3 tipos de operación principales (venta, alquiler, alquiler_temporal), y usamos percentiles P1/P99 por moneda+operación para cortar outliers extremos.

In [0]:
%sql
-- ============================================================
-- Estadísticas de PRECIO (filtrado)
-- Solo USD/ARS, operaciones principales, sin outliers (P1-P99)
-- ============================================================

with limites as (
    select
        moneda,
        tipo_de_operacion,
        percentile(precio, 0.01) as p01,
        percentile(precio, 0.99) as p99
    from propiedades_clean 
    where precio > 0
      and moneda in ('USD', 'ARS')
      and tipo_de_operacion in ('venta', 'alquiler')
    group by moneda, tipo_de_operacion
)
select 
    p.moneda,
    p.tipo_de_operacion,
    count(*) as cantidad,
    round(min(p.precio), 2) as precio_minimo,
    round(max(p.precio), 2) as precio_maximo,
    round(avg(p.precio), 2) as precio_promedio,
    round(percentile(p.precio, 0.05), 2) as precio_mediana,
    round(percentile(p.precio, 0.25), 2) as precio_p25,
    round(percentile(p.precio, 0.75), 2) as precio_p75
from propiedades_clean p
join limites l
 on p.moneda = l.moneda
 and p.tipo_de_operacion = l.tipo_de_operacion
where p.precio between l.p01 and l.p99
group by p.moneda, p.tipo_de_operacion
order by cantidad desc;

In [0]:
%sql
create or replace temporary view propiedades_clean_2 as (
    with limites as (
    select
        moneda,
        tipo_de_operacion,
        percentile(precio, 0.01) as p01,
        percentile(precio, 0.99) as p99
    from propiedades_clean 
    where precio > 0
      and moneda in ('USD', 'ARS')
      and tipo_de_operacion in ('venta', 'alquiler')
    group by moneda, tipo_de_operacion
)
select 
    p.*
from propiedades_clean p 
join limites l
 on p.moneda = l.moneda
 and p.tipo_de_operacion = l.tipo_de_operacion
where p.precio between l.p01 and l.p99
);

In [0]:
%sql
-- ============================================================
-- Estadísticas de METROS CUADRADOS
-- ============================================================

select 
    'metros_cuadrados_totales' as columna,
    count(*) as cantidad_no_nulos,
    round(min(metros_cuadrados_totales), 2) as minimo,
    round(max(metros_cuadrados_totales), 2) as maximo,
    round(avg(metros_cuadrados_totales), 2) as promedio,
    round(percentile(metros_cuadrados_totales, 0.05), 2) as mediana
from propiedades_clean_2
where metros_cuadrados_totales is not null and metros_cuadrados_totales > 0

union all 

select 
    'metros_cuadrados_cubiertos' as columna,
    count(*) as cantidad_no_nulos,
    round(min(metros_cuadrados_cubiertos), 2) as minimo,
    round(max(metros_cuadrados_cubiertos), 2) as maximo,
    round(avg(metros_cuadrados_cubiertos), 2) as promedio,
    round(percentile(metros_cuadrados_cubiertos, 0.5), 2) as mediana
from propiedades_clean_2
where metros_cuadrados_cubiertos is not null and metros_cuadrados_cubiertos > 0;

In [0]:
%sql
-- ============================================================
-- Análisis de ANTIGÜEDAD
-- ============================================================
-- NOTA: Vimos en los datos que 999 parece ser un valor por defecto (missing)


select 
    antiguedad,
    count(*) as cantidad_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from propiedades_clean_2
group by antiguedad
order by cantidad_registros desc
limit 20;


In [0]:
%sql
-- ============================================================
-- Estadísticas de antigüedad SIN el valor 999
-- ============================================================

select 
    count(*) as cantidad,
    min(cast(antiguedad as int)) as antiguedad_minima,
    max(cast(antiguedad as int)) as antiguedad_maxima,
    round(avg(cast(antiguedad as int)), 2) as antiguedad_promedio,
    round(percentile(cast(antiguedad as int), 0.5), 2) as mediana
from propiedades_clean_2 
where antiguedad != '999' 
  and antiguedad is not null 
  and antiguedad >= 0;


In [0]:
%sql
-- ============================================================
-- Reporte de calidad de datos usando CTEs
-- ============================================================


with total as (
    select 
        count(*) as n
    from propiedades_clean_2
),
problemas as (
    select 
        -- Precios problemáticos
        count(case when precio is null or precio <= 0 then 1 end) as precios_invalidos,
        -- Metros cuadrados problemáticos
        count(case when metros_cuadrados_totales is null or metros_cuadrados_totales <= 0 then 1 end) as m2_invalidos,
        -- Antigüedad con valor 999 (placeholder)
        count(case when antiguedad = '999' or antiguedad = 999 then 1 end) as antiguedad_999,
        -- Ambientes = 0 o NULL
        count(case when ambientes = 0 or ambientes is null then 1 end) as ambientes_invalidos,
        -- Moneda vacía
        count(case when moneda is null or moneda = '' then 1 end) as moneda_vacia,
        -- Tipo de operación vacía
        count(case when tipo_de_operacion is null or tipo_de_operacion = '' then 1 end) as tipo_de_operacion_vacia
    from propiedades_clean_2

)
select 
    t.n as total_registros,
    p.precios_invalidos,
    round(p.precios_invalidos * 100.0 / t.n, 2) as pct_precios_invalidos,
    p.m2_invalidos,
    round(p.m2_invalidos * 100.0 / t.n, 2) as pct_m2_invalidos,
    p.antiguedad_999,
    round(p.antiguedad_999 * 100.0 / t.n, 2) as pct_antiguedad_999,
    p.ambientes_invalidos,
    round(p.ambientes_invalidos * 100.0 / t.n, 2) as pct_ambientes_invalidos,
    p.moneda_vacia,
    round(p.moneda_vacia * 100.0 / t.n, 2) as pct_moneda_vacia,
    p.tipo_de_operacion_vacia,
    round(p.tipo_de_operacion_vacia * 100.0 / t.n, 2) as pct_tipo_de_operacion_vacia
from total t, problemas p; 


In [0]:
%sql
-- ============================================================
-- Detectar posibles duplicados
-- ============================================================

with duplicados as (
select 
    precio,
    url,
    count(*) as veces
from real_estate.bronze.properties_bronze
group by precio, url
having count(*) > 1
)
select 
    count(*) as grupos_duplicados,
    sum(veces) as total_registros_duplicados,
    sum(veces - 1) registros_extra_duplicados
from duplicados;


In [0]:
%sql
-- ============================================================
-- Ver ejemplos de duplicados
-- ============================================================

with duplicados as (
select 
    precio,
    url,
    count(*) as veces
from real_estate.bronze.properties_bronze
group by precio, url
having count(*) > 1
)
select *
from duplicados
order by veces desc
limit 10;

In [0]:
%sql
-- ============================================================
-- Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
with stats as (
select 
    moneda,
    round(percentile(precio, 0.01), 2) as p01,
    round(percentile(precio, 0.99), 2) as p99
from propiedades_clean_2
where precio > 0
group by moneda
)
select 
    p.moneda,
    'Muy bojo (< P01)' as tipo_outlier,
    count(*) as cantidad
from propiedades_clean_2 p
join stats s
on p.moneda = s.moneda
where p.precio < s.p01 and p.precio > 0
group by p.moneda

union all

select 
    p.moneda,
    'Muy alto (> p99)' as tipo_outlier,
    count(*) as cantidad
from propiedades_clean_2 p
join stats s 
on p.moneda = s.moneda
where p.precio > s.p99
group by p.moneda
order by moneda, tipo_outlier


In [0]:
%sql

with zonas as (
    select 
        zona,
        round(avg(precio), 2) as precio_promedio,
        count(*) as cantidad_propiedades
    from propiedades_clean_2
    where precio > 0
      and zona is not null
      and tipo_de_operacion = 'alquiler'
      and moneda = 'ARS'
    group by zona
    having count(*) >= 50
),
zonas_rankeadas as(
select 
    zona,
    precio_promedio,
    cantidad_propiedades as propiedades,
    row_number() over (order by precio_promedio) as ranking
from zonas
)
select 
    ranking,
    zona,
    precio_promedio,
    propiedades
from zonas_rankeadas
where ranking <= 10
order by ranking